## Part 1 — The state  ·  **TODO 1**

The state is the graph's memory: one dictionary that every node reads from and writes to. Declaring
it as a `TypedDict` is how you say *these* are the fields an alert carries as it moves through the
graph.

This is the first real difference from the LangChain module. There, data flowed along a pipe from
one step to the next. Here every node sees the **whole** state, and a node only returns the keys it
changed.

> **Expected output:** `state fields: ['alert', 'risk', 'cve_info', 'report']`.

In [1]:
import csv
import json
import tempfile
from pathlib import Path
from typing import TypedDict
from importlib.metadata import version
from IPython.display import display, HTML

# Keep the original practice data outside the final five-file project folder.
PRACTICE_DIR = Path(tempfile.mkdtemp(prefix="langgraph_practice_"))
ALERTS_FILE = PRACTICE_DIR / "alerts.json"
ALERTS_FILE.write_text(json.dumps({'about': "12 labelled security alerts for Mini-Project 4. `expected` is the routing decision a SOC analyst would make: 'escalate' for anything a human must look at now, 'close' for routine noise. Your agent's job is to reproduce these decisions. 6 of each. Do NOT edit this file — build your own set in my_alerts.json (copy alerts_template.json).", 'alerts': [{'id': 'A01', 'alert': 'Critical RCE exploit attempt against the internet-facing Apache server', 'expected': 'escalate', 'note': 'textbook high severity'}, {'id': 'A02', 'alert': 'Ransomware canary file modified on the finance file share', 'expected': 'escalate', 'note': 'textbook high severity'}, {'id': 'A03', 'alert': 'Large outbound data transfer to an unknown host at 03:14', 'expected': 'escalate', 'note': "data theft, but the word 'exfiltration' never appears"}, {'id': 'A04', 'alert': 'New local administrator account created on the domain controller', 'expected': 'escalate', 'note': 'privilege escalation'}, {'id': 'A05', 'alert': 'Credential dumping tool signature detected in memory on HR-LAPTOP-07', 'expected': 'escalate', 'note': 'serious, but uses none of the obvious keywords'}, {'id': 'A06', 'alert': 'Repeated MFA prompts followed by a successful login from a new country', 'expected': 'escalate', 'note': 'MFA fatigue attack; reads mild, is not'}, {'id': 'A07', 'alert': 'Scheduled vulnerability scan reported critical findings on the sandbox VM', 'expected': 'close', 'note': "says 'critical' but it is our own scanner on a throwaway host"}, {'id': 'A08', 'alert': 'User reset their own password via the self-service portal', 'expected': 'close', 'note': 'routine'}, {'id': 'A09', 'alert': 'Antivirus quarantined an EICAR test file during a scheduled test', 'expected': 'close', 'note': 'routine, deliberately triggered'}, {'id': 'A10', 'alert': 'Single failed login from a known corporate device', 'expected': 'close', 'note': 'routine'}, {'id': 'A11', 'alert': 'Backup job completed with a warning about a skipped temp folder', 'expected': 'close', 'note': 'routine'}, {'id': 'A12', 'alert': 'Administrator ran an approved patch deployment during the maintenance window', 'expected': 'close', 'note': "says 'administrator' but it is change-managed and approved"}]}, indent=2), encoding="utf-8")

# TODO 1 completed: all fields passed between the nodes.
class TriageState(TypedDict):
    alert: str
    risk: str
    cve_info: str
    report: str

print("LangGraph:", version("langgraph"))
print("State fields:", list(TriageState.__annotations__))

LangGraph: 1.2.17
State fields: ['alert', 'risk', 'cve_info', 'report']


## Part 2 — The classify node  ·  **TODO 2**

A node is just a Python function: it takes the state and returns a dict of the keys it changed.

This particular node is the one your whole experiment turns on. It decides `risk` by looking for
words in the alert text — the same rule as the Lecture 3 notebook. Keep `KEYWORDS` as a variable:
Part 7 changes it and re-measures.

> **Expected output:** `high` for the RCE alert, `low` for the failed-login alert.

In [2]:
KEYWORDS = ["rce", "ransomware", "exfil", "critical", "admin"]

# TODO 2 completed: return just the changed state fields.
def classify(state: TriageState) -> dict:
    hit = any(word in state["alert"].lower() for word in KEYWORDS)
    return {"risk": "high" if hit else "low"}

for text in ["Critical RCE on public server", "Single failed login"]:
    print(text, "->", classify({"alert": text}))

Critical RCE on public server -> {'risk': 'high'}
Single failed login -> {'risk': 'low'}


## Part 3 — The router  ·  **TODO 3**

This is what makes it a graph rather than a chain. A **router** is a function that looks at the
state and returns a *label* — the name of the branch to take next. LangGraph calls it after the
`classify` node and jumps to whichever node that label points at.

The router decides nothing else: no work, no side effects, just "which way".

> **Expected output:** `high -> investigate`, `low -> quick_close`.

In [3]:
# TODO 3 completed: return a route label, not a new state.
def route_by_risk(state: TriageState) -> str:
    return "investigate" if state["risk"] == "high" else "quick_close"

print("High risk ->", route_by_risk({"risk": "high"}))
print("Low risk ->", route_by_risk({"risk": "low"}))

High risk -> investigate
Low risk -> quick_close


## Part 4 — A node that calls a tool  ·  **TODO 4**

A **tool** is just a function the agent can call for information it does not have. `lookup_cve` is
given — a tiny stand-in for a threat-intel API. Your job is the node that uses it and writes the
result into the state.

> **Expected output:** the RCE alert returns the `CVE-2026-0001` line; an unmatched alert returns
> the "No specific CVE found" line.

In [4]:
# Course mock data only. CVE-2026-0001 is not verified intelligence.
def lookup_cve(keyword: str) -> str:
    db = {
        "rce": "CVE-2026-0001: critical RCE; patch immediately.",
        "ransomware": "Multiple CVEs; isolate the host and restore from backup.",
        "credential": "Credential theft; force password resets and review logins.",
        "outbound": "Possible data theft; check egress logs and block the destination.",
    }
    for word, info in db.items():
        if word in keyword.lower():
            return info
    return "No specific CVE found; follow standard triage."

# TODO 4 completed: a node calls a tool and returns its result in state.
def investigate(state: TriageState) -> dict:
    info = lookup_cve(state["alert"])
    return {"cve_info": info, "report": "ESCALATE. " + info}

def quick_close(state: TriageState) -> dict:
    return {"cve_info": "", "report": "CLOSE. Logged for monitoring; no action needed."}

print(investigate({"alert": "Critical RCE on a server"}))

{'cve_info': 'CVE-2026-0001: critical RCE; patch immediately.', 'report': 'ESCALATE. CVE-2026-0001: critical RCE; patch immediately.'}


## Part 5 — Build and compile the graph  ·  **TODO 5**

Now wire the pieces together. The nodes are added for you; you supply the three lines that make it
a runnable graph — where it starts, how the branch works, and the compile step.

```
        classify
           |  route_by_risk
     +-----+-----+
     |           |
investigate   quick_close
     |           |
     +-----+-----+
           END
```

> **Expected output:** the RCE alert reports `ESCALATE. CVE-2026-0001 ...`; the failed-login alert
> reports `CLOSE. Logged for monitoring ...`.

In [5]:
from langgraph.graph import StateGraph, END

def build_agent():
    builder = StateGraph(TriageState)
    builder.add_node("classify", classify)
    builder.add_node("investigate", investigate)
    builder.add_node("quick_close", quick_close)
    # TODO 5 completed: entry, conditional edge, final edges, and compile.
    builder.set_entry_point("classify")
    builder.add_conditional_edges("classify", route_by_risk, {
        "investigate": "investigate", "quick_close": "quick_close"
    })
    builder.add_edge("investigate", END)
    builder.add_edge("quick_close", END)
    return builder.compile()

agent = build_agent()

def run_agent(text):
    return agent.invoke({"alert": text, "risk": "", "cve_info": "", "report": ""})

print(run_agent("Ransomware detected on a file server"))

{'alert': 'Ransomware detected on a file server', 'risk': 'high', 'cve_info': 'Multiple CVEs; isolate the host and restore from backup.', 'report': 'ESCALATE. Multiple CVEs; isolate the host and restore from backup.'}


## Part 6 — Score the agent  *(no TODO — run it and read it)*

The coding is done. Everything from here is measurement and interpretation.

`alerts.json` holds **12 real-looking alerts**, each labelled with what a SOC analyst would
actually do: **6 `escalate`** and **6 `close`**. Comparing your agent's route to that label gives
three numbers — and the two errors are not equally bad:

| | Meaning | Cost |
|---|---|---|
| **caught** | escalated something that deserved it | what you want |
| **missed** | closed a real incident | an attacker keeps going, unnoticed |
| **over-escalated** | escalated routine noise | analyst time, and eventually alert fatigue |

> **Expected output with the Lecture 3 keyword list: caught 3/6, missed 3/6, over-escalated 2/6.**
> The agent you just built misses **half** of the real incidents. Part 7 asks what to do about it.

In [6]:
PROVIDED_ALERTS = json.loads(ALERTS_FILE.read_text(encoding="utf-8"))["alerts"]

def score_practice():
    counts = {"caught": 0, "missed": 0, "over_escalated": 0}
    for item in PROVIDED_ALERTS:
        state = run_agent(item["alert"])
        actual = "escalate" if state["risk"] == "high" else "close"
        if item["expected"] == "escalate":
            counts["caught" if actual == "escalate" else "missed"] += 1
        elif actual == "escalate":
            counts["over_escalated"] += 1
    return counts

print("Provided alerts:", len(PROVIDED_ALERTS))
print("Five-keyword practice baseline:", score_practice())
print("Caught and missed are out of 6 incidents; over_escalated is out of 6 routine alerts.")

Provided alerts: 12
Five-keyword practice baseline: {'caught': 3, 'missed': 3, 'over_escalated': 2}
Caught and missed are out of 6 incidents; over_escalated is out of 6 routine alerts.


## Part 7 — Tune the rule and watch the two errors trade  *(no TODO)*

The obvious fix for a missed incident is to add words. Do that four times over and watch what it
costs.

> **Expected output:**
>
> | keyword list | caught | missed | over-escalated |
> |---|---:|---:|---:|
> | narrow (2 words) | 2/6 | **4/6** | 0/6 |
> | **lecture (5 words)** | 3/6 | **3/6** | 2/6 |
> | wider (8 words) | **6/6** | 0/6 | 2/6 |
> | widest (11 words) | 6/6 | 0/6 | **4/6** |
>
> Read the two error columns against each other. Going from narrow to widest drives missed
> detections to zero and doubles the false escalations. There is no row with zeroes in both.
>
> **And look at *which* alerts stay wrong.** A07 and A12 are over-escalated by every list that
> catches all six real incidents. A07 says *critical* — about findings from our own scanner on a
> sandbox VM. A12 says *Administrator* — about an approved patch in a maintenance window. The
> trigger word is right there in both; what makes them harmless is **context the words do not
> carry**. No keyword list can fix that, which is the most important thing this table has to say.

In [7]:
KEYWORD_LISTS = {
    "narrow2": ["rce", "ransomware"],
    "lecture5": ["rce", "ransomware", "exfil", "critical", "admin"],
    "wider8": ["rce", "ransomware", "exfil", "critical", "admin", "outbound", "credential", "mfa"],
    "widest11": ["rce", "ransomware", "exfil", "critical", "admin", "outbound", "credential", "mfa", "failed", "scan", "password"],
}
practice_rows = []
for name, words in KEYWORD_LISTS.items():
    KEYWORDS[:] = words
    row = {"rule": name, **score_practice()}
    practice_rows.append(row)
    print(row)
KEYWORDS[:] = KEYWORD_LISTS["lecture5"]
# This practice CSV is temporary; results_mine.csv is the submitted table.
with (PRACTICE_DIR / "results.csv").open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(practice_rows[0]))
    writer.writeheader()
    writer.writerows(practice_rows)

{'rule': 'narrow2', 'caught': 2, 'missed': 4, 'over_escalated': 0}
{'rule': 'lecture5', 'caught': 3, 'missed': 3, 'over_escalated': 2}
{'rule': 'wider8', 'caught': 6, 'missed': 0, 'over_escalated': 2}
{'rule': 'widest11', 'caught': 6, 'missed': 0, 'over_escalated': 4}


## Part 8 — Memory: why the graph beats a chain  *(no TODO)*

One more LangGraph feature from Lecture 3. Compile the **same** graph with a `MemorySaver` and every
run tagged with a `thread_id` is remembered, so an incident can be picked up later — something a
one-shot chain cannot do.

> **Expected output:** the state saved under `incident-42` is still readable after the run, showing
> the alert, the risk, and the report the agent produced.

In [8]:
from langgraph.checkpoint.memory import InMemorySaver

practice_builder = StateGraph(TriageState)
practice_builder.add_node("classify", classify)
practice_builder.add_node("investigate", investigate)
practice_builder.add_node("quick_close", quick_close)
practice_builder.set_entry_point("classify")
practice_builder.add_conditional_edges("classify", route_by_risk, {
    "investigate": "investigate", "quick_close": "quick_close"
})
practice_builder.add_edge("investigate", END)
practice_builder.add_edge("quick_close", END)
practice_memory = practice_builder.compile(checkpointer=InMemorySaver())
practice_cfg = {"configurable": {"thread_id": "incident-42"}}
practice_memory.invoke({"alert": "Ransomware detected on a file server", "risk": "", "cve_info": "", "report": ""}, practice_cfg)
print("Saved incident-42:", practice_memory.get_state(practice_cfg).values)
print("Unused incident-99:", practice_memory.get_state({"configurable": {"thread_id": "incident-99"}}).values)

Saved incident-42: {'alert': 'Ransomware detected on a file server', 'risk': 'high', 'cve_info': 'Multiple CVEs; isolate the host and restore from backup.', 'report': 'ESCALATE. Multiple CVEs; isolate the host and restore from backup.'}
Unused incident-99: {}


## Part 9A Read and explain my twelve alerts

The first six need analyst review; the last six are routine in their stated circumstances. M05 is the incident without any tested keyword. M07 is an approved lab test that contains `ransomware`. M06 is a second hard incident, involving a mailbox rule. Public references and the fictional context are documented in SOURCES.md.

The additional fields are simulated log data, not answer labels. The graph receives the text and those fields. The scoring function receives `expected` only after the graph has made a decision.

In [9]:
from collections import Counter
from html import escape

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "my_alerts.json").is_file() and (PROJECT_DIR / "langgraph_project/my_alerts.json").is_file():
    PROJECT_DIR = PROJECT_DIR / "langgraph_project"
assert (PROJECT_DIR / "my_alerts.json").is_file(), "Put my_alerts.json beside the notebook, or open the notebook from langgraph_project."
MY_ALERTS = json.loads((PROJECT_DIR / "my_alerts.json").read_text(encoding="utf-8"))["alerts"]
labels = Counter(a["expected"] for a in MY_ALERTS)
assert len(MY_ALERTS) == 12 and len({a["id"] for a in MY_ALERTS}) == 12
assert labels == {"escalate": 6, "close": 6}, "Keep six examples of each label."
BY_ID = {a["id"]: a for a in MY_ALERTS}
assert not any(w in BY_ID["M05"]["alert"].lower() for w in KEYWORD_LISTS["widest11"])
assert "ransomware" in BY_ID["M07"]["alert"].lower()

def show_table(rows, columns=None):
    columns = columns or list(rows[0])
    head = "".join(f"<th style='padding:6px;text-align:left'>{escape(str(c))}</th>" for c in columns)
    body = "".join("<tr>" + "".join(f"<td style='padding:6px;border-top:1px solid #ddd;vertical-align:top'>{escape(str(row.get(c, '')))}</td>" for c in columns) + "</tr>" for row in rows)
    display(HTML(f"<table style='border-collapse:collapse'><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>"))

print("My alerts:", len(MY_ALERTS), "| Labels:", dict(labels))
show_table(MY_ALERTS, ["id", "alert", "expected", "note"])

My alerts: 12 | Labels: {'escalate': 6, 'close': 6}


id,alert,expected,note
M01,EDR detected ransomware rewriting finance documents on FIN-LT-04 while the employee was using the laptop.,escalate,Files are being changed on a production laptop during normal use; this needs an immediate analyst review.
M02,A critical RCE alert on WEB-01 was followed by an unexpected process launched by the public web service.,escalate,The unexpected child process adds evidence that the exposed service may have been exploited.
M03,"Cloud audit logs show an admin role granted to a regular user without an approved change, and the user denies requesting it.",escalate,"An unapproved increase in access needs investigation, especially when the user denies the action."
M04,"Repeated MFA prompts ended with a sign-in from an unfamiliar device, and the employee says they did not approve it.",escalate,Repeated prompts plus an unfamiliar successful session suggest account misuse.
M05,Successful VPN session for a departed employee; no current owner approved the activity.,escalate,This account belongs to someone who left the organization. A successful session needs review even though no tested keyword appears.
M06,A mailbox rule now sends copies of payroll messages to an unfamiliar personal inbox; the owner denies creating it.,escalate,"The rule could expose payroll information. None of the tested keywords appears, and the current context tool does not check forwarding destinations."
M07,Ransomware test detection on LAB-WS-01 matched an authorized exercise; no production files were touched.,close,"The lab target, simulation event, and time match approved record CHG-1042. This is the routine alert that contains a keyword."
M08,Administrator installed a signed update on APP-02 during the approved maintenance window.,close,"The target, patch event, and time match approved record CHG-1043; this is expected maintenance."
M09,"One failed sign-in on a managed laptop was followed by a successful sign-in from the same device, and the employee confirmed a typing mistake.",close,The single failure is explained by the user and followed by normal activity; there are no other suspicious signals in this example.
M10,The employee completed a password reset through the normal self-service page and confirmed the request with the help desk.,close,The user confirmed a routine reset through the approved process.


## Part 9B State and the new context lookup tool

State is the case information passed between nodes. Each node returns just the fields it changes. The read-only tool checks a small fictional account directory and two approved changes. Approval requires the correct asset, account, event type, status, and time window. Alert wording alone cannot approve a change.

The directory and change records do not contain `expected` or `note`. The tool is ordinary Python, called by the `context_lookup` node. It does not use a model, contact a real system, disable an account, or isolate a computer.

In [10]:
from datetime import datetime

class MyState(TypedDict, total=False):
    alert_id: str
    alert: str
    asset: str
    account: str
    event_type: str
    change_id: str
    time_utc: str
    matched_words: list[str]
    risk: str
    context: dict
    decision: str
    reason: str
    report: str
    analyst_note: str
    review_count: int

ACCOUNT_DIRECTORY = {
    "former.staff": {"status": "departed"},
    **{name: {"status": "active"} for name in [
        "finance.user", "web.service", "cloud.user", "sales.user", "payroll.user",
        "lab.tester", "patch.service", "office.user", "backup.service"
    ]},
}
APPROVED_CHANGES = {
    "CHG-1042": {"status": "approved", "asset": "LAB-WS-01", "account": "lab.tester",
                 "event_type": "edr_simulation", "start": "2026-10-20T10:00:00+00:00", "end": "2026-10-20T11:00:00+00:00"},
    "CHG-1043": {"status": "approved", "asset": "APP-02", "account": "patch.service",
                 "event_type": "patch_install", "start": "2026-10-20T10:00:00+00:00", "end": "2026-10-20T12:00:00+00:00"},
}

def lookup_context(asset, account, event_type, time_utc, change_id):
    record = APPROVED_CHANGES.get(change_id)
    approved = False
    if record:
        moment = datetime.fromisoformat(time_utc)
        approved = (
            record["status"] == "approved"
            and record["asset"] == asset
            and record["account"] == account
            and record["event_type"] == event_type
            and datetime.fromisoformat(record["start"]) <= moment <= datetime.fromisoformat(record["end"])
        )
    return {
        "account_status": ACCOUNT_DIRECTORY.get(account, {}).get("status", "unknown"),
        "approved_change": bool(approved),
        "matched_change": change_id if approved else "",
    }

INPUT_FIELDS = ("alert", "asset", "account", "event_type", "change_id", "time_utc")
def graph_input(item):
    # Explicit allowlist: the reference label and rationale never reach the agent.
    return {"alert_id": item["id"], **{k: item[k] for k in INPUT_FIELDS}, "analyst_note": "", "review_count": 0}

assert not {"expected", "note"} & graph_input(MY_ALERTS[0]).keys()
print("M05 context:", lookup_context(**{k: BY_ID["M05"][k] for k in INPUT_FIELDS if k != "alert"}))
print("M07 context:", lookup_context(**{k: BY_ID["M07"][k] for k in INPUT_FIELDS if k != "alert"}))

M05 context: {'account_status': 'departed', 'approved_change': False, 'matched_change': ''}
M07 context: {'account_status': 'active', 'approved_change': True, 'matched_change': 'CHG-1042'}


## Part 9C Build the baseline and the changed graph

`keywords` mode routes directly after `classify`. `approval` mode inserts a `context_lookup` node and routes after that lookup. `context` mode uses the same added node and also escalates a sign-in by a departed account. That final rule takes priority over any approval.

The list of keywords stays fixed while these structural/routing changes are compared. A conditional edge chooses one destination; it does not run both branches. `END` finishes the run. The terminal node saves a reason and a report; "close" here is a predicted label, not a real action on a security system.

In [11]:
def build_my_graph(words, mode="keywords", checkpointer=None):
    words = tuple(words)  # Each configuration keeps its own words.
    assert mode in {"keywords", "approval", "context"}

    def classify_mine(state):
        matches = [word for word in words if word in state["alert"].lower()]
        return {"matched_words": matches, "risk": "high" if matches else "low"}

    def context_node(state):
        context = lookup_context(**{k: state[k] for k in INPUT_FIELDS if k != "alert"})
        return {"context": context}

    def decision_and_reason(state):
        context = state.get("context", {}) if mode != "keywords" else {}
        if mode == "context" and state["event_type"] == "sign_in" and context.get("account_status") == "departed":
            return "escalate", "Successful sign-in uses an account marked departed."
        if mode != "keywords" and context.get("approved_change"):
            return "close", "Exact match to approved change " + context["matched_change"] + "."
        if state["risk"] == "high":
            return "escalate", "Matched keywords: " + ", ".join(state["matched_words"]) + "."
        return "close", "No tested keyword or enabled context rule requested escalation."

    def route(state):
        decision, _ = decision_and_reason(state)
        return "investigate" if decision == "escalate" else "quick_close"

    def finish(state):
        decision, reason = decision_and_reason(state)
        return {"decision": decision, "reason": reason,
                "report": decision.upper() + ". " + reason,
                "review_count": state.get("review_count", 0) + 1}

    builder = StateGraph(MyState)
    builder.add_node("classify", classify_mine)
    builder.add_node("investigate", finish)
    builder.add_node("quick_close", finish)
    builder.set_entry_point("classify")
    route_from = "classify"
    if mode != "keywords":
        builder.add_node("context_lookup", context_node)
        builder.add_edge("classify", "context_lookup")
        route_from = "context_lookup"
    builder.add_conditional_edges(route_from, route, {
        "investigate": "investigate", "quick_close": "quick_close"
    })
    builder.add_edge("investigate", END)
    builder.add_edge("quick_close", END)
    return builder.compile(checkpointer=checkpointer)

baseline_graph = build_my_graph(KEYWORD_LISTS["wider8"], "keywords")
selected_graph = build_my_graph(KEYWORD_LISTS["wider8"], "context")
for name, graph in [("Keyword graph", baseline_graph), ("Changed graph", selected_graph)]:
    print(name)
    for edge in graph.get_graph().edges:
        print(" ", edge.source, "->", edge.target, "(conditional)" if edge.conditional else "")

Keyword graph
  __start__ -> classify 
  classify -> investigate (conditional)
  classify -> quick_close (conditional)
  investigate -> __end__ 
  quick_close -> __end__ 
Changed graph
  __start__ -> classify 
  classify -> context_lookup 
  context_lookup -> investigate (conditional)
  context_lookup -> quick_close (conditional)
  investigate -> __end__ 
  quick_close -> __end__ 


## Part 9D Measure one change at a time

**Caught:** expected escalate and predicted escalate. **Missed:** expected escalate but predicted close. **Over-escalated:** expected close but predicted escalate. Caught and missed each use the six incident alerts; over-escalated uses the six routine alerts. The counts stay separate.

Each row names the comparison row and the single setting that changed. The lecture word list is kept as the baseline. The same twelve examples are used for every configuration; this is tuning on a small development set, not an independent test.

In [12]:
CONFIGS = [
    {"configuration": "keyword_lecture", "profile": "lecture5", "mode": "keywords", "compare_to": "", "change": "reference: course five-word rule"},
    {"configuration": "keyword_narrow", "profile": "narrow2", "mode": "keywords", "compare_to": "keyword_lecture", "change": "keyword list only"},
    {"configuration": "keyword_wider", "profile": "wider8", "mode": "keywords", "compare_to": "keyword_lecture", "change": "keyword list only"},
    {"configuration": "keyword_widest", "profile": "widest11", "mode": "keywords", "compare_to": "keyword_wider", "change": "keyword list only"},
    {"configuration": "approval_wider", "profile": "wider8", "mode": "approval", "compare_to": "keyword_wider", "change": "add context node and approval route"},
    {"configuration": "context_wider", "profile": "wider8", "mode": "context", "compare_to": "approval_wider", "change": "enable departed-account sign-in rule"},
    {"configuration": "context_widest", "profile": "widest11", "mode": "context", "compare_to": "context_wider", "change": "keyword list only"},
]

def evaluate(graph):
    counts = {"caught": 0, "missed": 0, "over_escalated": 0}
    details = []
    for item in MY_ALERTS:
        state = graph.invoke(graph_input(item))
        actual = state["decision"]
        if item["expected"] == "escalate":
            outcome = "caught" if actual == "escalate" else "missed"
            counts[outcome] += 1
        else:
            outcome = "over_escalated" if actual == "escalate" else "correct_close"
            if outcome == "over_escalated":
                counts[outcome] += 1
        details.append({"id": item["id"], "expected": item["expected"], "actual": actual,
                        "outcome": outcome, "reason": state["reason"]})
    assert counts["caught"] + counts["missed"] == 6
    return counts, details

experiment_rows, DETAILS = [], {}
for config in CONFIGS:
    words = KEYWORD_LISTS[config["profile"]]
    graph = build_my_graph(words, config["mode"])
    counts, details = evaluate(graph)
    DETAILS[config["configuration"]] = details
    row = {**config, "keyword_count": len(words), "keywords": "|".join(words), **counts,
           "incident_total": 6, "routine_total": 6,
           "missed_ids": "|".join(d["id"] for d in details if d["outcome"] == "missed"),
           "over_escalated_ids": "|".join(d["id"] for d in details if d["outcome"] == "over_escalated")}
    experiment_rows.append(row)

with (PROJECT_DIR / "results_mine.csv").open("w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(experiment_rows[0]))
    writer.writeheader()
    writer.writerows(experiment_rows)
show_table(experiment_rows, ["configuration", "compare_to", "change", "caught", "missed", "over_escalated"])
print("Saved results_mine.csv")

configuration,compare_to,change,caught,missed,over_escalated
keyword_lecture,,reference: course five-word rule,3,3,2
keyword_narrow,keyword_lecture,keyword list only,2,4,1
keyword_wider,keyword_lecture,keyword list only,4,2,3
keyword_widest,keyword_wider,keyword list only,4,2,5
approval_wider,keyword_wider,add context node and approval route,4,2,1
context_wider,approval_wider,enable departed-account sign-in rule,5,1,1
context_widest,context_wider,keyword list only,5,1,3


Saved results_mine.csv


In [13]:
SELECTED = "context_wider"
selected_result = next(row for row in experiment_rows if row["configuration"] == SELECTED)
print("Selected:", SELECTED)
print({key: selected_result[key] for key in ["caught", "missed", "over_escalated", "missed_ids", "over_escalated_ids"]})
show_table(DETAILS[SELECTED])
missed_sets = [{d["id"] for d in details if d["outcome"] == "missed"} for details in DETAILS.values()]
print("Missed in every configuration:", sorted(set.intersection(*missed_sets)))
print("One alert is 1/6 of its class, or about 16.7 percentage points.")

Selected: context_wider
{'caught': 5, 'missed': 1, 'over_escalated': 1, 'missed_ids': 'M06', 'over_escalated_ids': 'M12'}


id,expected,actual,outcome,reason
M01,escalate,escalate,caught,Matched keywords: ransomware.
M02,escalate,escalate,caught,"Matched keywords: rce, critical."
M03,escalate,escalate,caught,Matched keywords: admin.
M04,escalate,escalate,caught,Matched keywords: mfa.
M05,escalate,escalate,caught,Successful sign-in uses an account marked departed.
M06,escalate,close,missed,No tested keyword or enabled context rule requested escalation.
M07,close,close,correct_close,Exact match to approved change CHG-1042.
M08,close,close,correct_close,Exact match to approved change CHG-1043.
M09,close,close,correct_close,No tested keyword or enabled context rule requested escalation.
M10,close,close,correct_close,No tested keyword or enabled context rule requested escalation.


Missed in every configuration: ['M06']
One alert is 1/6 of its class, or about 16.7 percentage points.


## Part 9E Live traces for the video

Run these cells while recording. `stream_mode="updates"` shows the actual node updates; the cumulative display is the state after each node. The router reads that state and picks the next node. `risk` records the keyword result; the final `decision` can change after context is checked.

M01 is a clear catch. M05 shows the added context turning a keyword miss into an escalation. M06 shows the selected graph still missing a mailbox incident. M07 shows why an exact approval record matters.

In [14]:
def trace_alert(alert_id, graph=selected_graph):
    item = BY_ID[alert_id]
    state = graph_input(item)
    print("ALERT", alert_id, "|", item["alert"])
    print("INPUT:", json.dumps(state, indent=2))
    for update in graph.stream(state, stream_mode="updates"):
        for node, changed in update.items():
            state.update(changed)
            print("\nNODE:", node)
            print("CHANGED:", json.dumps(changed, indent=2))
            print("STATE:", json.dumps(state, indent=2))
    print("\nEXPECTED:", item["expected"], "| ACTUAL:", state["decision"])
    print("WHY THE LABEL:", item["note"])
    return state

caught_example = trace_alert("M01")

ALERT M01 | EDR detected ransomware rewriting finance documents on FIN-LT-04 while the employee was using the laptop.
INPUT: {
  "alert_id": "M01",
  "alert": "EDR detected ransomware rewriting finance documents on FIN-LT-04 while the employee was using the laptop.",
  "asset": "FIN-LT-04",
  "account": "finance.user",
  "event_type": "file_encryption",
  "change_id": "",
  "time_utc": "2026-10-20T10:30:00+00:00",
  "analyst_note": "",
  "review_count": 0
}

NODE: classify
CHANGED: {
  "matched_words": [
    "ransomware"
  ],
  "risk": "high"
}
STATE: {
  "alert_id": "M01",
  "alert": "EDR detected ransomware rewriting finance documents on FIN-LT-04 while the employee was using the laptop.",
  "asset": "FIN-LT-04",
  "account": "finance.user",
  "event_type": "file_encryption",
  "change_id": "",
  "time_utc": "2026-10-20T10:30:00+00:00",
  "analyst_note": "",
  "review_count": 0,
  "matched_words": [
    "ransomware"
  ],
  "risk": "high"
}

NODE: context_lookup
CHANGED: {
  "context"

In [15]:
print("M05 before context:", baseline_graph.invoke(graph_input(BY_ID["M05"]))["report"])
context_example = trace_alert("M05")

M05 before context: CLOSE. No tested keyword or enabled context rule requested escalation.
ALERT M05 | Successful VPN session for a departed employee; no current owner approved the activity.
INPUT: {
  "alert_id": "M05",
  "alert": "Successful VPN session for a departed employee; no current owner approved the activity.",
  "asset": "VPN-GW",
  "account": "former.staff",
  "event_type": "sign_in",
  "change_id": "",
  "time_utc": "2026-10-20T10:30:00+00:00",
  "analyst_note": "",
  "review_count": 0
}

NODE: classify
CHANGED: {
  "matched_words": [],
  "risk": "low"
}
STATE: {
  "alert_id": "M05",
  "alert": "Successful VPN session for a departed employee; no current owner approved the activity.",
  "asset": "VPN-GW",
  "account": "former.staff",
  "event_type": "sign_in",
  "change_id": "",
  "time_utc": "2026-10-20T10:30:00+00:00",
  "analyst_note": "",
  "review_count": 0,
  "matched_words": [],
  "risk": "low"
}

NODE: context_lookup
CHANGED: {
  "context": {
    "account_status": "

In [16]:
missed_example = trace_alert("M06")

ALERT M06 | A mailbox rule now sends copies of payroll messages to an unfamiliar personal inbox; the owner denies creating it.
INPUT: {
  "alert_id": "M06",
  "alert": "A mailbox rule now sends copies of payroll messages to an unfamiliar personal inbox; the owner denies creating it.",
  "asset": "MAIL-SERVICE",
  "account": "payroll.user",
  "event_type": "mailbox_rule",
  "change_id": "",
  "time_utc": "2026-10-20T10:30:00+00:00",
  "analyst_note": "",
  "review_count": 0
}

NODE: classify
CHANGED: {
  "matched_words": [],
  "risk": "low"
}
STATE: {
  "alert_id": "M06",
  "alert": "A mailbox rule now sends copies of payroll messages to an unfamiliar personal inbox; the owner denies creating it.",
  "asset": "MAIL-SERVICE",
  "account": "payroll.user",
  "event_type": "mailbox_rule",
  "change_id": "",
  "time_utc": "2026-10-20T10:30:00+00:00",
  "analyst_note": "",
  "review_count": 0,
  "matched_words": [],
  "risk": "low"
}

NODE: context_lookup
CHANGED: {
  "context": {
    "accoun

In [17]:
print("M07 before context:", baseline_graph.invoke(graph_input(BY_ID["M07"]))["report"])
approval_example = trace_alert("M07")

M07 before context: ESCALATE. Matched keywords: ransomware.
ALERT M07 | Ransomware test detection on LAB-WS-01 matched an authorized exercise; no production files were touched.
INPUT: {
  "alert_id": "M07",
  "alert": "Ransomware test detection on LAB-WS-01 matched an authorized exercise; no production files were touched.",
  "asset": "LAB-WS-01",
  "account": "lab.tester",
  "event_type": "edr_simulation",
  "change_id": "CHG-1042",
  "time_utc": "2026-10-20T10:30:00+00:00",
  "analyst_note": "",
  "review_count": 0
}

NODE: classify
CHANGED: {
  "matched_words": [
    "ransomware"
  ],
  "risk": "high"
}
STATE: {
  "alert_id": "M07",
  "alert": "Ransomware test detection on LAB-WS-01 matched an authorized exercise; no production files were touched.",
  "asset": "LAB-WS-01",
  "account": "lab.tester",
  "event_type": "edr_simulation",
  "change_id": "CHG-1042",
  "time_utc": "2026-10-20T10:30:00+00:00",
  "analyst_note": "",
  "review_count": 0,
  "matched_words": [
    "ransomware"
 

## Part 9F Show memory with two thread IDs

This is a separate graph used only for the memory demo. The experiment above starts fresh for every alert, so previous cases do not affect its counts. With the same thread ID, a partial update keeps the old alert and context and adds the analyst note. A different thread starts empty. `review_count` shows that the same case ran again.

The saver below keeps state only in this Python process. Restarting the kernel removes it. A thread ID selects a case; it is not an access-control system.

In [18]:
memory_graph = build_my_graph(KEYWORD_LISTS["wider8"], "context", checkpointer=InMemorySaver())
case_a = {"configurable": {"thread_id": "case-M05"}}
case_b = {"configurable": {"thread_id": "case-M01"}}

first = memory_graph.invoke(graph_input(BY_ID["M05"]), case_a)
print("First visit:", first["alert_id"], "review_count =", first["review_count"])
second = memory_graph.invoke({"analyst_note": "Checked the directory; the account is still marked departed."}, case_a)
saved = memory_graph.get_state(case_a).values
print("Same thread:", saved["alert_id"], "review_count =", saved["review_count"])
print("Saved note:", saved["analyst_note"])
print("Saved decision:", saved["decision"])
print("Different thread before use:", memory_graph.get_state(case_b).values)
other = memory_graph.invoke(graph_input(BY_ID["M01"]), case_b)
print("Different case:", other["alert_id"], "review_count =", other["review_count"], "note =", repr(other["analyst_note"]))
print("Original case still has its note:", memory_graph.get_state(case_a).values["analyst_note"])

assert first["review_count"] == 1 and saved["review_count"] == 2
assert saved["alert_id"] == "M05" and other["alert_id"] == "M01"
assert other["analyst_note"] == ""

First visit: M05 review_count = 1
Same thread: M05 review_count = 2
Saved note: Checked the directory; the account is still marked departed.
Saved decision: escalate
Different thread before use: {}
Different case: M01 review_count = 1 note = ''
Original case still has its note: Checked the directory; the account is still marked departed.
